In [1]:

import pandas as pd
import numpy as np

# 1. Load dataset
file_path = "Amazon Sale Report.csv"
df = pd.read_csv(file_path)

# 2. Basic information
print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)
print("Shape:", df.shape)
print("Duplicate full rows:", df.duplicated().sum())
print("Unique Order IDs:", df["Order ID"].nunique())

# 3. Column names, data types and missing values
print("\n" + "=" * 60)
print("2. COLUMN QUALITY REPORT")
print("=" * 60)

quality = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2),
    "Unique_Values": df.nunique(dropna=True)
})

print(quality.to_string())

# Save the report
quality.to_csv("Amazon_Column_Quality_Report.csv")

# 4. Check blank strings and extra spaces
print("\n" + "=" * 60)
print("3. BLANK AND WHITESPACE VALUES")
print("=" * 60)

for col in df.select_dtypes(include="object").columns:
    blank_count = df[col].astype("string").str.strip().eq("").sum()
    if blank_count > 0:
        print(f"{col}: {blank_count} blank values")

# 5. Check date conversion
print("\n" + "=" * 60)
print("4. DATE VALIDATION")
print("=" * 60)

parsed_dates = pd.to_datetime(
    df["Date"], format="%m-%d-%y", errors="coerce"
)

print("Unparseable dates:", parsed_dates.isna().sum())
print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())

# 6. Check numeric columns
print("\n" + "=" * 60)
print("5. NUMERIC VALIDATION")
print("=" * 60)

for col in ["Qty", "Amount", "ship-postal-code"]:
    numeric = pd.to_numeric(df[col], errors="coerce")
    print(f"\n{col}")
    print("Invalid values:", (numeric.isna() & df[col].notna()).sum())
    print("Negative values:", (numeric < 0).sum())
    print("Zero values:", (numeric == 0).sum())
    print("Minimum:", numeric.min())
    print("Maximum:", numeric.max())

# 7. Inspect missing amounts by order status
print("\n" + "=" * 60)
print("6. MISSING AMOUNTS BY STATUS")
print("=" * 60)

print(
    df.groupby("Status", dropna=False)
      .agg(
          Total_Rows=("Order ID", "size"),
          Missing_Amount=("Amount", lambda x: x.isna().sum()),
          Missing_Courier=("Courier Status", lambda x: x.isna().sum())
      )
      .to_string()
)

# 8. Check repeated order IDs (not automatically duplicates)
print("\n" + "=" * 60)
print("7. REPEATED ORDER IDs")
print("=" * 60)

print("Rows belonging to repeated Order IDs:",
      df["Order ID"].duplicated(keep=False).sum())

# 9. Check category, status, city and country consistency
print("\n" + "=" * 60)
print("8. DISTINCT TEXT VALUES")
print("=" * 60)

for col in [
    "Status", "Fulfilment", "Sales Channel",
    "Category", "Size", "Courier Status",
    "currency", "ship-country", "B2B"
]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(25).to_string())

# 10. Save a small sample for inspection
df.head(100).to_csv("Amazon_Sales_Sample.csv", index=False)

print("\n" + "=" * 60)
print("AUDIT COMPLETE")
print("Files created:")
print("- Amazon_Column_Quality_Report.csv")
print("- Amazon_Sales_Sample.csv")
print("=" * 60)


1. DATASET OVERVIEW
Shape: (128975, 21)
Duplicate full rows: 0
Unique Order IDs: 120378

2. COLUMN QUALITY REPORT
                   Data_Type  Missing_Count  Missing_Percent  Unique_Values
Index                  int64              0             0.00         128975
Order ID                 str              0             0.00         120378
Date                     str              0             0.00             91
Status                   str              0             0.00             13
Fulfilment               str              0             0.00              2
Sales Channel            str              0             0.00              2
ship-service-level       str              0             0.00              2
Style                    str              0             0.00           1377
SKU                      str              0             0.00           7195
Category                 str              0             0.00              9
Size                     str              0       

C:\Users\Shashanki Rawat\AppData\Local\Temp\ipykernel_1020\3201932011.py:38: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:



4. DATE VALIDATION
Unparseable dates: 55109
Earliest date: 2022-03-31 00:00:00
Latest date: 2022-06-29 00:00:00

5. NUMERIC VALIDATION

Qty
Invalid values: 0
Negative values: 0
Zero values: 12807
Minimum: 0
Maximum: 15

Amount
Invalid values: 0
Negative values: 0
Zero values: 2343
Minimum: 0.0
Maximum: 5584.0

ship-postal-code
Invalid values: 0
Negative values: 0
Zero values: 0
Minimum: 110001.0
Maximum: 989898.0

6. MISSING AMOUNTS BY STATUS
                               Total_Rows  Missing_Amount  Missing_Courier
Status                                                                    
Cancelled                           18332            7566             6861
Pending                               658               2                0
Pending - Waiting for Pick Up         281               0                0
Shipped                             77804             208                0
Shipped - Damaged                       1               0                0
Shipped - Delivered to Buye

KeyError: 'Sales Channel'

In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. LOAD DATA
# ==========================================

file_path = Path("Amazon Sale Report.csv")
df = pd.read_csv(file_path)

original_rows = len(df)

print("Original shape:", df.shape)

# ==========================================
# 2. STANDARDIZE COLUMN NAMES
# ==========================================

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("\nStandardized columns:")
print(df.columns.tolist())

# ==========================================
# 3. REMOVE UNNECESSARY INDEX COLUMN
# ==========================================

# Drop only if Index is just a sequential CSV row number
if "index" in df.columns:
    index_values = pd.to_numeric(df["index"], errors="coerce")
    if index_values.equals(
        pd.Series(np.arange(len(df)), index=df.index, dtype="float64")
    ):
        df.drop(columns=["index"], inplace=True)
        print("\nRemoved redundant index column.")

# ==========================================
# 4. STANDARDIZE TEXT AND BLANK VALUES
# ==========================================

text_columns = df.select_dtypes(
    include=["object", "str", "string"]
).columns.tolist()

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# Standardize casing for selected categorical columns
for col in ["Category", "Fulfilment", "Sales Channel",
            "ship-service-level", "Courier Status",
            "currency", "ship-country"]:
    if col in df.columns:
        df[col] = df[col].str.strip()

# ==========================================
# 5. CONVERT DATE SAFELY
# ==========================================

if "Date" in df.columns:
    original_dates = df["Date"].copy()

    # Try parsing mixed date formats
    parsed_dates = pd.to_datetime(
        original_dates,
        format="mixed",
        errors="coerce",
        dayfirst=False
    )

    # Compare an alternative interpretation when useful
    alternative_dates = pd.to_datetime(
        original_dates,
        format="mixed",
        errors="coerce",
        dayfirst=True
    )

    if alternative_dates.notna().sum() > parsed_dates.notna().sum():
        parsed_dates = alternative_dates

    df["Date"] = parsed_dates

    print("\nDate validation:")
    print("Original non-missing dates:", original_dates.notna().sum())
    print("Successfully parsed dates:", df["Date"].notna().sum())
    print("Unparsed dates:", df["Date"].isna().sum())
    print("Date range:", df["Date"].min(), "to", df["Date"].max())

# ==========================================
# 6. CONVERT NUMERIC COLUMNS
# ==========================================

numeric_columns = ["Qty", "Amount", "ship-postal-code"]

for col in numeric_columns:
    if col in df.columns:
        original = df[col].copy()
        converted = pd.to_numeric(original, errors="coerce")

        invalid_mask = original.notna() & converted.isna()
        print(f"\n{col} invalid values:", invalid_mask.sum())

        if col == "Qty":
            # Preserve missing values while allowing nullable integers
            df[col] = converted.astype("Int64")
        else:
            df[col] = converted

# Postal codes are identifiers, not quantities.
# Convert them to text while preserving missing values.
if "ship-postal-code" in df.columns:
    df["ship-postal-code"] = (
        df["ship-postal-code"]
        .map(lambda x: str(int(x)) if pd.notna(x) and float(x).is_integer()
             else str(x) if pd.notna(x) else pd.NA)
        .astype("string")
    )

# ==========================================
# 7. INVESTIGATE SUSPICIOUS VALUES
# ==========================================

print("\nZero quantity:", (df["Qty"] == 0).sum())
print("Negative quantity:", (df["Qty"] < 0).sum())
print("Zero amount:", (df["Amount"] == 0).sum())
print("Negative amount:", (df["Amount"] < 0).sum())

# Do not automatically replace zero amounts or quantities.
# Do not invent missing sales amounts.

# ==========================================
# 8. CHECK DUPLICATES
# ==========================================

duplicate_count = df.duplicated().sum()
print("\nExact duplicate rows:", duplicate_count)

# Remove exact duplicate rows only if they exist
if duplicate_count > 0:
    df = df.drop_duplicates().copy()

# Repeated order IDs can be legitimate multiple-item orders.
# Do not remove them.

# ==========================================
# 9. SAVE CLEANED DATASET
# ==========================================

output_file = "Amazon_Sales_Cleaned.csv"
df.to_csv(output_file, index=False)

# ==========================================
# 10. SAVE DATA QUALITY REPORT
# ==========================================

quality_report = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2),
    "Unique_Values": df.nunique(dropna=True)
})

quality_report.to_csv("Amazon_Cleaning_Quality_Report.csv")

# Save rows needing further investigation
issues = pd.Series(False, index=df.index)

if "Date" in df.columns:
    issues |= df["Date"].isna()

if "Qty" in df.columns:
    issues |= df["Qty"].isna() | (df["Qty"] <= 0)

if "Amount" in df.columns:
    issues |= df["Amount"].isna() | (df["Amount"] < 0)

df.loc[issues].to_csv("Amazon_Rows_To_Review.csv", index=False)

# ==========================================
# 11. FINAL SUMMARY
# ==========================================

print("\n" + "=" * 50)
print("DATA CLEANING COMPLETED")
print("=" * 50)
print("Original rows:", original_rows)
print("Final rows:", len(df))
print("Final columns:", len(df.columns))
print("Remaining exact duplicates:", df.duplicated().sum())
print("Total missing values:", int(df.isna().sum().sum()))

print("\nFiles created:")
print("1. Amazon_Sales_Cleaned.csv")
print("2. Amazon_Cleaning_Quality_Report.csv")
print("3. Amazon_Rows_To_Review.csv")

print("\nFinal data types:")
print(df.dtypes)


Original shape: (128975, 21)

Standardized columns:
['Index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'B2B']

Date validation:
Original non-missing dates: 128975
Successfully parsed dates: 128975
Unparsed dates: 0
Date range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00

Qty invalid values: 0

Amount invalid values: 0

ship-postal-code invalid values: 0

Zero quantity: 12807
Negative quantity: 0
Zero amount: 2343
Negative amount: 0

Exact duplicate rows: 0

DATA CLEANING COMPLETED
Original rows: 128975
Final rows: 128975
Final columns: 21
Remaining exact duplicates: 0
Total missing values: 22594

Files created:
1. Amazon_Sales_Cleaned.csv
2. Amazon_Cleaning_Quality_Report.csv
3. Amazon_Rows_To_Review.csv

Final data types:
Index                          int64
Order ID                      strin